# 10 · Kafka, from nothing

You do not need to have heard of Kafka.

Every cell below is **real Kafka code that you run**. Nothing is hidden in a
helper file. By the end you will have sent a message to a broker running on this
laptop, found it again by its address, and read it back.

---

## Word 1 · an **event**

An event is **a thing that happened**. Past tense. Finished.

```
    a rider requested a ride at 14:02
    a driver accepted it   at 14:04
    the ride completed     at 14:31
```

Three facts about an event:

**It is in the past.** Not a plan. It happened.
**It cannot be changed.** You cannot un-happen it. To correct it you send a
*new* event; you never edit the old one.
**It is small.** One thing, one moment.

Now compare a database row. A row is a statement about **the present**: *"this
ride currently has status completed"*. Update it and the previous value is gone.
Nothing remembers it used to say `in_progress`.

> A table tells you **how things are**.
> A stream of events tells you **how things got that way**.

---

## Word 2 · the **broker**, and Word 3 · a **topic**

The broker is a program that holds events. A topic is a named place inside it.

![](img/kafka-1-shape.png)

There is a broker running in Docker on this laptop. Let us ask it what it has.

In [ ]:
import sys; sys.path.insert(0, '.')
from nb import run                       # the one helper: runs a pipeline, shows its log
from pipelines.lib.config import KAFKA   # the address, read from .env

from confluent_kafka.admin import AdminClient

BROKER = KAFKA                           # usually localhost:19092
print('broker:', BROKER)

admin = AdminClient({'bootstrap.servers': BROKER})
metadata = admin.list_topics(timeout=10)

for name, topic in sorted(metadata.topics.items()):
    if name.startswith('__'):
        continue                     # Kafka's own internal bookkeeping
    print(f'{name:28} {len(topic.partitions)} partitions')

Four topics. They exist for the same reason folders exist: so different kinds
of thing do not get mixed up. Ride events in one, GPS pings in another.

`.dlq` means **dead letter queue**, where a message goes when nothing can read
it.

---

## Word 4 · a **producer**

A producer is anything that sends events. The KERB app is one. You are about to
be one.

Here is the smallest possible producer. Read it before you run it.

In [ ]:
from confluent_kafka import Producer
from pipelines.lib.config import TOPIC_RIDES
import json, datetime as dt

TOPIC = TOPIC_RIDES                      # 'kerb.trips.lifecycle'

# 1. a producer is a client object. It knows one thing: where the broker is.
producer = Producer({'bootstrap.servers': BROKER})

# 2. the event itself. Just a dictionary. Kafka does not care what is in it.
event = {
    'trip_id':   'TRP-DEMO-001',
    'event':     'requested',
    'ts':        dt.datetime.now(dt.timezone.utc).replace(microsecond=0).isoformat(),
    'driver_id': None,
    'pu_zone_id': 42,
}

# 3. Kafka speaks bytes, not python. So we serialise to JSON and encode.
value = json.dumps(event).encode('utf-8')
key   = event['trip_id'].encode('utf-8')     # why a key matters comes later

# 4. produce() does not send immediately. It queues.
producer.produce(TOPIC, key=key, value=value)

# 5. flush() waits until everything queued has actually reached the broker.
#    Forget this line and your program can exit before the message is sent.
producer.flush(10)

print('sent')

### Five lines, and the last one catches people out

`produce()` queues. `flush()` is what actually waits for the broker to confirm.
A script that produces and then exits without flushing sends nothing at all, and
raises no error.

## But where did it go?

The broker tells you, if you ask. Add a delivery callback.

In [ ]:
landed = {}

def on_delivery(err, msg):
    """Kafka calls this once it knows what happened to the message."""
    if err is not None:
        print('failed:', err)
    else:
        landed['partition'] = msg.partition()
        landed['offset']    = msg.offset()

event['trip_id'] = 'TRP-DEMO-002'
producer.produce(TOPIC,
                 key=event['trip_id'].encode(),
                 value=json.dumps(event).encode(),
                 on_delivery=on_delivery)      # <- the callback
producer.flush(10)

print(f"it landed at partition {landed['partition']}, offset {landed['offset']}")

---

## Word 5 · a **partition**, and Word 6 · an **offset**

Those two numbers are the message's permanent address.

![](img/kafka-2-partitions.png)

A topic is not one queue. It is split, so several readers can work at once.

Let us count what is actually on each partition.

In [ ]:
from confluent_kafka import Consumer, TopicPartition

consumer = Consumer({'bootstrap.servers': BROKER,
                     'group.id': 'just-looking'})     # any name, we only read metadata

partitions = sorted(consumer.list_topics(TOPIC, timeout=10).topics[TOPIC].partitions)

total = 0
print(f'{"partition":>10} {"first":>10} {"next":>10} {"messages":>12}')
for p in partitions:
    # watermarks: the lowest offset still kept, and the offset the NEXT message gets
    low, high = consumer.get_watermark_offsets(TopicPartition(TOPIC, p), timeout=10)
    total += high - low
    print(f'{p:>10} {low:>10,} {high:>10,} {high - low:>12,}')
print(f'{"total":>10} {"":>10} {"":>10} {total:>12,}')
consumer.close()

### The rule that matters

**Kafka only guarantees order *within* one partition.** Across partitions there
is no order at all.

So how do five events of one ride stay in order? Because we passed a **key**
(`trip_id`), and Kafka always puts the same key on the same partition.

> Same key → same partition → same order. Always.

Get that wrong and `completed` can land before `requested`, with no error,
because nothing is broken. You asked for the impossible.

---

## Reading is not destructive

Go back to the exact address of the message you sent and read it again.

In [ ]:
consumer = Consumer({'bootstrap.servers': BROKER,
                     'group.id': 'just-looking',
                     'enable.auto.commit': False})

# assign() means "put me at this exact position", ignoring any saved bookmark
tp = TopicPartition(TOPIC, landed['partition'], landed['offset'])
consumer.assign([tp])

msg = None
for _ in range(20):                # poll until the message arrives
    m = consumer.poll(1.0)
    if m is not None and not m.error():
        msg = m
        break
consumer.close()

print(f'read from partition {msg.partition()}, offset {msg.offset()}:\n')
print(json.dumps(json.loads(msg.value()), indent=2))

### It is still there

**A queue is destructive**: take a message off and it is gone, and only one
consumer can ever have it.

**Kafka is a log, not a queue.** Reading removes nothing. Five different teams
read the same events independently, and none of them asks the others'
permission.

---

## Word 7 · a **consumer group**

If reading removes nothing, how does a consumer remember where it got to?

It tells the broker, under a name. That name is the consumer group.

![](img/kafka-3-groups.png)

In [ ]:
groups = admin.list_consumer_groups(request_timeout=10).result().valid

for g in sorted(x.group_id for x in groups):
    c = Consumer({'bootstrap.servers': BROKER, 'group.id': g,
                  'enable.auto.commit': False})
    tps = [TopicPartition(TOPIC, p) for p in partitions]
    position = lag = 0
    for tp, committed in zip(tps, c.committed(tps, timeout=10)):
        low, high = c.get_watermark_offsets(tp, timeout=10)
        # a group that has never committed has offset -1001, so fall back to low
        at = committed.offset if committed.offset and committed.offset >= 0 else low
        position += at
        lag      += high - at
    c.close()
    print(f'{g:24} position {position:>10,}   still to read {lag:>8,}')

Two groups read this topic. `teach-bronze-events` is our pipeline.
`bronze-loader` belongs to a different project entirely.

**Separate bookmarks. Neither can affect the other.**

That last number is called **lag**: how far behind a group is.

It also explains something you will hit: `reset` has to delete the group,
because the bookmark lives on the **broker**, not in our database. Drop our
tables without deleting the group and the pipeline wakes up believing it has
already read everything.

---

## Word 8 · a **consumer**

Now write one. This is the shape of every Kafka consumer you will ever write.

We start it **before** sending anything, so you can watch events arrive live.

In [ ]:
from confluent_kafka import Consumer

watcher = Consumer({
    'bootstrap.servers': BROKER,
    'group.id': 'notebook-watcher',    # our name. the broker keeps our bookmark under it
    'auto.offset.reset': 'latest',     # never read before? start at the END, not the beginning
    'enable.auto.commit': False,       # WE decide when the bookmark moves
})

# subscribe means "give me this topic, wherever my bookmark is"
watcher.subscribe([TOPIC])

# subscribing is a request, not an answer. The broker has to assign us partitions,
# and that only happens while we poll. So poll until we actually have them.
for _ in range(20):
    watcher.poll(1.0)
    if watcher.assignment():
        break

print('watching:', sorted(f'partition {t.partition}' for t in watcher.assignment()))
print('nothing to read yet. it is sitting at the end of the topic, waiting.')

### subscribe, versus assign

Two different asks, and the difference matters.

| | |
|---|---|
| `assign()` | *put me at exactly this partition and offset.* We used it earlier to go back to one message. No group, no bookmark. |
| `subscribe()` | *give me this topic, and remember where I got to.* The broker hands out the partitions and keeps our place. |

---

## Now send a whole ride

Five events, one ride. Same key every time, so they all land on the same
partition, in order.

In [ ]:
import random

trip_id = f'TRP-DEMO-{random.randint(1000, 9999)}'
now     = dt.datetime.now(dt.timezone.utc).replace(microsecond=0)
driver  = f'DRV{random.randint(1, 2800):06d}'
fare    = round(random.uniform(60, 420), 2)

LIFECYCLE = ['requested', 'accepted', 'driver_arrived', 'started', 'completed']

producer = Producer({'bootstrap.servers': BROKER})
for i, name in enumerate(LIFECYCLE):
    e = {'trip_id': trip_id,
         'event':   name,
         'ts':      (now + dt.timedelta(minutes=i * 3)).isoformat(),
         'driver_id': None if name == 'requested' else driver,
         'pu_zone_id': 42}
    if name == 'completed':
        e['fare'] = fare
    producer.produce(TOPIC, key=trip_id.encode(), value=json.dumps(e).encode())
    print(f'  queued {name}')
producer.flush(10)
print(f'\n{trip_id}: 5 events on the topic')

## The watcher was listening. Ask it what it saw.

In [ ]:
seen = []
for _ in range(20):
    batch = watcher.consume(num_messages=500, timeout=1.0)
    for m in batch or []:
        if m.error():
            continue
        d = json.loads(m.value())
        if d.get('trip_id') == trip_id:              # only the ride we just sent
            seen.append((m.partition(), m.offset(), d['event']))
    if len(seen) == 5:
        break

for part, off, ev in seen:
    print(f'  partition {part}  offset {off:>8}  {ev}')

**All five, on the same partition, in the order they happened.**

That is the key doing its job. Had we passed no key, Kafka would have spread the
five events across all three partitions, and `completed` could have been read
before `requested`, with no error, because nothing would be broken. We would
have asked for the impossible.

## We read them. We have not saved our place.

`enable.auto.commit` was `False`, so the bookmark has not moved. Prove that,
then move it deliberately.

In [ ]:
tps = watcher.assignment()

before = watcher.committed(tps, timeout=10)
print('bookmark before commit:', [t.offset for t in before], '  (-1001 means never set)')

watcher.commit(asynchronous=False)          # NOW save our place

after = watcher.committed(tps, timeout=10)
print('bookmark after  commit:', [t.offset for t in after])
watcher.close()

### This is the whole safety argument

Read a batch. **Write it.** *Then* commit.

Crash before the commit and the next run reads the same messages again, and
writes them again. That is called **at-least-once**, and it is why every table
we write has a primary key and an `ON CONFLICT DO NOTHING`: writing the same
event twice has to be harmless.

Turn auto-commit on and you get the opposite order. Kafka saves your place while
the rows are still only in memory. Crash there and those messages are gone for
good, and nothing anywhere reports an error.

---

## The events exist. The warehouse has never heard of them.

Two questions, and both get answered with a real number rather than a promise.

### 1 · Where exactly are our five events?

We already know, because the watcher told us. Say it out loud.

In [ ]:
for part, off, ev in seen:
    print(f'  partition {part}  offset {off:>8}  {ev}')

our_partition = seen[0][0]
first_offset  = min(o for _, o, _ in seen)
last_offset   = max(o for _, o, _ in seen)

print(f'\nall five are on partition {our_partition}, offsets {first_offset:,} to {last_offset:,}')

### 2 · How far has the pipeline read on that partition?

The pipeline is a consumer group called `teach-bronze-events`. Its bookmark
lives on the broker. Ask the broker where it is.

In [ ]:
def bookmark(group, partition):
    """Where a group has read up to on one partition, and where the end is."""
    c = Consumer({'bootstrap.servers': BROKER, 'group.id': group,
                  'enable.auto.commit': False})
    tp = TopicPartition(TOPIC, partition)
    committed  = c.committed([tp], timeout=10)[0]
    low, high  = c.get_watermark_offsets(tp, timeout=10)
    c.close()
    # a group that has never committed reports -1001, so fall back to the oldest
    at = committed.offset if committed.offset and committed.offset >= 0 else low
    return at, high

at, high = bookmark('teach-bronze-events', our_partition)

print(f'partition {our_partition}')
print(f'  the pipeline has read up to offset  {at:>10,}')
print(f'  our first event sits at offset      {first_offset:>10,}')
print(f'  the newest message on it is at      {high - 1:>10,}')
print()
if at <= first_offset:
    print(f'  the bookmark is BEFORE our events. It has not seen them.')
    print(f'  {high - at:,} messages on this partition are still unread.')
else:
    print('  a scheduled run has already read past them. Send another ride and rerun.')

### And the answer that settles it

Offsets are Kafka's opinion. The warehouse is the thing finance queries. Ask it.

In [ ]:
import psycopg
from pipelines.lib.config import dsn, SCHEMA

with psycopg.connect(dsn()) as c:
    try:
        n_rows = c.execute(f'SELECT count(*) FROM {SCHEMA}.bronze_events WHERE trip_id = %s',
                           (trip_id,)).fetchone()[0]
    except Exception:
        n_rows = 0        # the table does not exist yet, which is also zero rows

print(f'SELECT count(*) FROM {SCHEMA}.bronze_events WHERE trip_id = {trip_id!r}')
print(f'\n  {n_rows}')
print('\nfive events exist on the topic. Nothing has moved them.')

---

## And now the real pipeline, which does exactly this plus the bookkeeping

Same producer, same consumer, same commit-last ordering. What it adds is the
contract, the quarantine, and a row in the run log.

In [ ]:
run('-m', 'pipelines.p2_bronze_events')

In [ ]:
at, high = bookmark('teach-bronze-events', our_partition)

print(f'partition {our_partition}')
print(f'  our last event was at offset  {last_offset:>10,}')
print(f'  the bookmark is now at        {at:>10,}   (the offset it will read NEXT)')
print(f'  so it has moved past all five of ours\n')

with psycopg.connect(dsn()) as c:
    rows = c.execute(f"""SELECT event, happened_at, fare
                        FROM {SCHEMA}.bronze_events WHERE trip_id = %s
                        ORDER BY happened_at""", (trip_id,)).fetchall()
print(f'\n{trip_id} in the warehouse:')
for e, t, f in rows:
    print(f'  {e:16} {t}  {f if f is not None else ""}')

---

## Last thing: a message nobody can read

The pipeline's contract requires `trip_id`, `event` and `ts`. Send one with an
empty `ts` and watch what happens to it, and to the four around it.

In [ ]:
broken_id = f'TRP-BROKEN-{random.randint(1000, 9999)}'
producer = Producer({'bootstrap.servers': BROKER})

for i, name in enumerate(LIFECYCLE):
    e = {'trip_id': broken_id, 'event': name,
         'ts': (now + dt.timedelta(minutes=i * 3)).isoformat(),
         'driver_id': driver}
    if name == 'started':
        e['ts'] = ''            # the contract requires this. it is empty.
    producer.produce(TOPIC, key=broken_id.encode(), value=json.dumps(e).encode())
producer.flush(10)
print(f'{broken_id}: 5 events sent, one of them unreadable')

In [ ]:
run('-m', 'pipelines.p2_bronze_events')

In [ ]:
with psycopg.connect(dsn()) as c:
    landed_n = c.execute(f'SELECT count(*) FROM {SCHEMA}.bronze_events WHERE trip_id = %s',
                         (broken_id,)).fetchone()[0]
    held = c.execute(f"""SELECT reason, payload->>'partition', payload->>'offset'
                        FROM {SCHEMA}.quarantine
                        WHERE payload->>'raw' LIKE %s""", (f'%{broken_id}%',)).fetchall()

print(f'landed in bronze_events : {landed_n}')
print(f'held in quarantine      : {len(held)}\n')
for reason, part, off in held:
    print(f'  {reason}')
    print(f'  it is still on the topic at partition {part}, offset {off}')

**Four landed. One held. Nothing crashed.**

And look at the last line: quarantine kept the **partition and offset**. Because
an offset is a permanent address, you can go straight back to the topic and read
that exact message again.

That is the difference between *"something went wrong last night"* and *"here it
is, this is the message, this is why"*.

---

## Every word, in one place

| | |
|---|---|
| **event** | a thing that happened. past tense, immutable |
| **broker** | the program that holds events. one is in Docker on this laptop |
| **topic** | a named place to put events |
| **producer** | anything that sends. `produce()` queues, `flush()` sends |
| **consumer** | anything that reads. reading removes nothing |
| **partition** | a topic is split, so several readers work at once |
| **offset** | a permanent address inside a partition |
| **key** | same key → same partition → order is kept |
| **consumer group** | a name, and a bookmark the broker stores against it |
| **lag** | how far behind a group's bookmark is |

## Three reasons Kafka exists

1. **A table says how things are. A stream says how they got that way.**
2. **Many independent consumers.** Reading does not remove.
3. **Partitions let readers work in parallel**, and a key keeps order where it matters.